In [11]:
import os
print(os.listdir("/kaggle/working/geomag"))

['CME_MAG_Implementation_Guide.md', 'eval.py', 'mag_pipeline.py', 'label_events.py', 'model_factory.py', 'train.py', 'feature_engineer.py', 'ensemble.py', '.git']


In [14]:
import sys
sys.path.insert(0, "/kaggle/working/geomag")

from data_pipeline import load_cdf_directory
print("import works!")

import works!


In [15]:
from mag_pipeline     import load_mag_directory, resample_mag_to_swis
from feature_engineer import build_feature_matrix, FEATURE_NAMES
from label_events     import attach_labels
from data_pipeline    import build_sequences, split_and_scale

import pandas as pd
import numpy as np

idx = pd.date_range("2024-05-01", periods=10_000, freq="1min")
rng = np.random.default_rng(0)

swis_raw = pd.DataFrame({
    "vsw": 400 + rng.normal(0, 30, len(idx)),
    "np":  6   + rng.normal(0, 2,  len(idx)),
    "tp":  1e5 * np.exp(rng.normal(0, 0.2, len(idx))),
    "he_flux": 0.3 + rng.normal(0, 0.05, len(idx)),
    "h_flux":  6   + rng.normal(0, 2,    len(idx)),
    "vth": rng.normal(50, 5, len(idx)),
}, index=idx)

mag_raw = pd.DataFrame({
    "bx": rng.normal(0, 5, len(idx)),
    "by": rng.normal(0, 5, len(idx)),
    "bz": rng.normal(0, 5, len(idx)),
    "b_mag": 7 + rng.normal(0, 2, len(idx)),
}, index=idx)

mag_aligned = resample_mag_to_swis(mag_raw, swis_index=swis_raw.index)
feat_df     = build_feature_matrix(swis_raw, mag_aligned)
feat_df["label"] = 0

X, y = build_sequences(feat_df, feature_cols=FEATURE_NAMES)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("All 14 features present:", X.shape[2] == 14)

2026-05-14 09:53:36 [INFO] mag_pipeline — MAG resampled to 10000 SWIS steps | NaN fraction: 0.0%
2026-05-14 09:53:36 [INFO] data_pipeline — Built 618 sequences (seq_len=128, stride=16) | CME rate=0.00%


Feature matrix: (10000, 14) | columns: ['vsw', 'np', 'tp', 'he_h_ratio', 'beta_proxy', 'dvsw_dt', 'bz', 'b_mag', 'clock_angle', 'dbz_dt', 'b_rotation', 'bz_smoothed', 'bz_persistence', 'b_elevation']
X shape: (618, 128, 14)
y shape: (618,)
All 14 features present: True


In [2]:
import os
import sys

# 1. Define your repo details
repo_url = "https://github.com/Aur1ety/Geomag-Detector.git"
repo_dir = "/kaggle/working/Geomag-Detector"

# 2. Clone or pull the latest code
if not os.path.exists(repo_dir):
    print("Cloning repository...")
    !git clone {repo_url}
else:
    print("Pulling latest changes...")
    %cd {repo_dir}
    !git pull
    %cd /kaggle/working

# 3. Add the repo to the system path so Python can import your modules
if repo_dir not in sys.path:
    sys.path.append(repo_dir)

# 4. Install the required space physics and ML libraries
print("Installing dependencies...")
!pip install cdflib xarray netCDF4 imbalanced-learn wandb -q

# 5. Check GPU status
import torch
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Pulling latest changes...
/kaggle/working/Geomag-Detector
Already up to date.
/kaggle/working
Installing dependencies...

GPU Available: True
Device: Tesla T4


In [26]:
!python /kaggle/working/geomag/train.py \
    --mag_dir "/kaggle/input/datasets/aurachan/mag-l1-data" \
    --epochs 100

2026-05-15 05:01:47,617 INFO Loading MAG data from: /kaggle/input/datasets/aurachan/mag-l1-data
2026-05-15 05:01:47,871 INFO Parsed: L2_AL1_MAG_20240509_V08.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 05:01:47,918 INFO Parsed: L2_AL1_MAG_20240510_V08.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 05:01:48,052 INFO Parsed: L2_AL1_MAG_20240511_V08.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 05:01:48,102 INFO Parsed: L2_AL1_MAG_20240512_V08.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 05:01:48,150 INFO Parsed: L2_AL1_MAG_20240513_V08.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 05:01:48,192 INFO Parsed: L2_AL1_MAG_20250701_V00.nc | 8640 rows | Bz [-7.4, 3.1] nT
2026-05-15 05:01:48,235 INFO Parsed: L2_AL1_MAG_20250702_V00.nc | 8640 rows | Bz [-7.5, 1.0] nT
2026-05-15 05:01:48,276 INFO Parsed: L2_AL1_MAG_20250703_V00.nc | 8640 rows | Bz [-15.6, 12.0] nT
2026-05-15 05:01:48,318 INFO Parsed: L2_AL1_MAG_20250704_V00.nc | 8640 rows | Bz [-11.1, 7.2] nT
2026-05-15 05:01:48,358 INFO Pa

In [27]:
import subprocess
subprocess.run(["git", "-C", "/kaggle/working/geomag", "pull"], check=True)

Updating a071c84..0b79a81
Fast-forward
 train.py | 10 +++++++---
 1 file changed, 7 insertions(+), 3 deletions(-)


From https://github.com/Aur1ety/Geomag-Detector
   a071c84..0b79a81  main       -> origin/main


CompletedProcess(args=['git', '-C', '/kaggle/working/geomag', 'pull'], returncode=0)

In [29]:
!python /kaggle/working/geomag/train.py \
    --mag_dir "/kaggle/input/datasets/aurachan/mag-l1-data" \
    --epochs 100

2026-05-15 05:44:29,910 INFO Loading MAG data from: /kaggle/input/datasets/aurachan/mag-l1-data
2026-05-15 05:44:30,092 INFO Parsed: L2_AL1_MAG_20240509_V08.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 05:44:30,135 INFO Parsed: L2_AL1_MAG_20240510_V08.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 05:44:30,177 INFO Parsed: L2_AL1_MAG_20240511_V08.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 05:44:30,213 INFO Parsed: L2_AL1_MAG_20240512_V08.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 05:44:30,253 INFO Parsed: L2_AL1_MAG_20240513_V08.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 05:44:30,292 INFO Parsed: L2_AL1_MAG_20250701_V00.nc | 8640 rows | Bz [-7.4, 3.1] nT
2026-05-15 05:44:30,330 INFO Parsed: L2_AL1_MAG_20250702_V00.nc | 8640 rows | Bz [-7.5, 1.0] nT
2026-05-15 05:44:30,369 INFO Parsed: L2_AL1_MAG_20250703_V00.nc | 8640 rows | Bz [-15.6, 12.0] nT
2026-05-15 05:44:30,407 INFO Parsed: L2_AL1_MAG_20250704_V00.nc | 8640 rows | Bz [-11.1, 7.2] nT
2026-05-15 05:44:30,447 INFO Pa

In [30]:
import os
def walk_all(path, depth=0):
    for item in os.listdir(path):
        full = os.path.join(path, item)
        print("  " * depth + item)
        if os.path.isdir(full) and depth < 5:
            walk_all(full, depth+1)

walk_all("/kaggle/input")

datasets
  aurachan
    mag-l1-data
      L2_AL1_MAG_20250713_V00.nc
      L2_AL1_MAG_20260214_V00.nc
      L2_AL1_MAG_20260122_V00.nc
      L2_AL1_MAG_20250719_V00.nc
      L2_AL1_MAG_20260203_V00.nc
      L2_AL1_MAG_20260128_V00.nc
      L2_AL1_MAG_20250729_V00.nc
      L2_AL1_MAG_20260123_V00.nc
      L2_AL1_MAG_20260126_V00.nc
      L2_AL1_MAG_20250727_V00.nc
      L2_AL1_MAG_20260209_V00.nc
      L2_AL1_MAG_20250709_V00.nc
      L2_AL1_MAG_20240509_V08.nc
      L2_AL1_MAG_20260205_V00.nc
      L2_AL1_MAG_20260130_V00.nc
      L2_AL1_MAG_20250710_V00.nc
      L2_AL1_MAG_20260201_V00.nc
      L2_AL1_MAG_20250708_V00.nc
      L2_AL1_MAG_20240511_V08.nc
      L2_AL1_MAG_20260204_V00.nc
      L2_AL1_MAG_20260215_V00.nc
      L2_AL1_MAG_20240510_V08.nc
      L2_AL1_MAG_20250716_V00.nc
      L2_AL1_MAG_20250722_V00.nc
      L2_AL1_MAG_20240513_V08.nc
      L2_AL1_MAG_20250721_V00.nc
      L2_AL1_MAG_20250725_V00.nc
      L2_AL1_MAG_20250703_V00.nc
      L2_AL1_MAG_20250728_V00.nc
      L

In [32]:
!python /kaggle/working/geomag/train.py \
    --mag_dir "/kaggle/input/datasets/aurachan/updated-windows-mag-data" \
    --epochs 100

2026-05-15 06:13:23,802 INFO Loading MAG data from: /kaggle/input/datasets/aurachan/updated-windows-mag-data
2026-05-15 06:13:24,039 INFO Parsed: L2_AL1_MAG_20240509_V00.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 06:13:24,087 INFO Parsed: L2_AL1_MAG_20240510_V00.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 06:13:24,134 INFO Parsed: L2_AL1_MAG_20240511_V00.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 06:13:24,171 INFO Parsed: L2_AL1_MAG_20240512_V00.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 06:13:24,214 INFO Parsed: L2_AL1_MAG_20240513_V00.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 06:13:24,260 INFO Parsed: L2_AL1_MAG_20241214_V00.nc | 8640 rows | Bz [-8.0, 10.6] nT
2026-05-15 06:13:24,307 INFO Parsed: L2_AL1_MAG_20241215_V00.nc | 8640 rows | Bz [-7.7, 8.3] nT
2026-05-15 06:13:24,353 INFO Parsed: L2_AL1_MAG_20241216_V00.nc | 8640 rows | Bz [-6.7, 8.0] nT
2026-05-15 06:13:24,398 INFO Parsed: L2_AL1_MAG_20241217_V00.nc | 8640 rows | Bz [-21.0, 29.0] nT
2026-05-15 06:13:2

In [45]:
!python /kaggle/working/geomag/train.py \
    --mag_dir "/kaggle/input/datasets/aurachan/updated-windows-mag-data" \
    --epochs 100

2026-05-15 06:28:10,067 INFO Loading MAG data from: /kaggle/input/datasets/aurachan/updated-windows-mag-data
2026-05-15 06:28:10,400 INFO Parsed: L2_AL1_MAG_20240509_V00.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 06:28:10,446 INFO Parsed: L2_AL1_MAG_20240510_V00.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 06:28:10,494 INFO Parsed: L2_AL1_MAG_20240511_V00.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 06:28:10,541 INFO Parsed: L2_AL1_MAG_20240512_V00.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 06:28:10,587 INFO Parsed: L2_AL1_MAG_20240513_V00.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 06:28:10,636 INFO Parsed: L2_AL1_MAG_20241214_V00.nc | 8640 rows | Bz [-8.0, 10.6] nT
2026-05-15 06:28:10,706 INFO Parsed: L2_AL1_MAG_20241215_V00.nc | 8640 rows | Bz [-7.7, 8.3] nT
2026-05-15 06:28:10,769 INFO Parsed: L2_AL1_MAG_20241216_V00.nc | 8640 rows | Bz [-6.7, 8.0] nT
2026-05-15 06:28:10,821 INFO Parsed: L2_AL1_MAG_20241217_V00.nc | 8640 rows | Bz [-21.0, 29.0] nT
2026-05-15 06:28:1

In [46]:

import subprocess
subprocess.run(["git", "-C", "/kaggle/working/geomag", "pull"], check=True)

Updating fbcf3d7..63c7072
Fast-forward
 model_factory.py | 225 ++++++++++++++++++++++++++++++++++++++++++++++++++++---
 train.py         | 189 ++++++++++++++++++++++++++++++++++++----------
 2 files changed, 364 insertions(+), 50 deletions(-)


From https://github.com/Aur1ety/Geomag-Detector
   fbcf3d7..63c7072  main       -> origin/main


CompletedProcess(args=['git', '-C', '/kaggle/working/geomag', 'pull'], returncode=0)

In [47]:
!python /kaggle/working/geomag/train.py \
    --mag_dir "/kaggle/input/datasets/aurachan/updated-windows-mag-data" \
    --epochs 100

2026-05-15 07:07:14,105 INFO Loading MAG data from: /kaggle/input/datasets/aurachan/updated-windows-mag-data
2026-05-15 07:07:14,427 INFO Parsed: L2_AL1_MAG_20240509_V00.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 07:07:14,474 INFO Parsed: L2_AL1_MAG_20240510_V00.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 07:07:14,521 INFO Parsed: L2_AL1_MAG_20240511_V00.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 07:07:14,560 INFO Parsed: L2_AL1_MAG_20240512_V00.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 07:07:14,602 INFO Parsed: L2_AL1_MAG_20240513_V00.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 07:07:14,647 INFO Parsed: L2_AL1_MAG_20241214_V00.nc | 8640 rows | Bz [-8.0, 10.6] nT
2026-05-15 07:07:14,688 INFO Parsed: L2_AL1_MAG_20241215_V00.nc | 8640 rows | Bz [-7.7, 8.3] nT
2026-05-15 07:07:14,731 INFO Parsed: L2_AL1_MAG_20241216_V00.nc | 8640 rows | Bz [-6.7, 8.0] nT
2026-05-15 07:07:14,775 INFO Parsed: L2_AL1_MAG_20241217_V00.nc | 8640 rows | Bz [-21.0, 29.0] nT
2026-05-15 07:07:1

In [48]:

import subprocess
subprocess.run(["git", "-C", "/kaggle/working/geomag", "pull"], check=True)

Updating 63c7072..1256f2c
Fast-forward
 model_factory.py | 58 +++++++++++++++++++++++---------------------------------
 train.py         | 15 ++++++++-------
 2 files changed, 32 insertions(+), 41 deletions(-)


From https://github.com/Aur1ety/Geomag-Detector
   63c7072..1256f2c  main       -> origin/main


CompletedProcess(args=['git', '-C', '/kaggle/working/geomag', 'pull'], returncode=0)

In [49]:
!python /kaggle/working/geomag/train.py \
    --mag_dir "/kaggle/input/datasets/aurachan/updated-windows-mag-data" \
    --epochs 100

2026-05-15 07:38:21,585 INFO Loading MAG data from: /kaggle/input/datasets/aurachan/updated-windows-mag-data
2026-05-15 07:38:21,951 INFO Parsed: L2_AL1_MAG_20240509_V00.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 07:38:21,998 INFO Parsed: L2_AL1_MAG_20240510_V00.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 07:38:22,046 INFO Parsed: L2_AL1_MAG_20240511_V00.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 07:38:22,084 INFO Parsed: L2_AL1_MAG_20240512_V00.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 07:38:22,128 INFO Parsed: L2_AL1_MAG_20240513_V00.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 07:38:22,178 INFO Parsed: L2_AL1_MAG_20241214_V00.nc | 8640 rows | Bz [-8.0, 10.6] nT
2026-05-15 07:38:22,226 INFO Parsed: L2_AL1_MAG_20241215_V00.nc | 8640 rows | Bz [-7.7, 8.3] nT
2026-05-15 07:38:22,273 INFO Parsed: L2_AL1_MAG_20241216_V00.nc | 8640 rows | Bz [-6.7, 8.0] nT
2026-05-15 07:38:22,318 INFO Parsed: L2_AL1_MAG_20241217_V00.nc | 8640 rows | Bz [-21.0, 29.0] nT
2026-05-15 07:38:2

In [60]:
import importlib, sys

# Force reload of all project modules
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["model_factory","feature_engineer",
                                "mag_pipeline","label_events","train"]):
        del sys.modules[mod]

import logging
sys.path.insert(0, "/kaggle/working/geomag")

import joblib
import numpy as np
# ... rest of the diagnostic cell
import sys, logging
sys.path.insert(0, "/kaggle/working/geomag")  # adjust if needed

import joblib
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from mag_pipeline     import load_mag_directory, resample_mag_to_1min
from feature_engineer import build_mag_features, MAG_FEATURE_NAMES
from label_events     import attach_labels
from model_factory    import PatchTransformer
from train            import build_sequences, SEQUENCE_LENGTH, STRIDE

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("diagnose")

MAG_DIR     = "/kaggle/input/datasets/aurachan/updated-windows-mag-data"
MODEL_PATH  = "saved_models/patchtransformer_mag_v1.pth"
SCALER_PATH = "/kaggle/working/scalers/scaler_mag.pkl"
THRESHOLD   = 0.88
OUT_DIR     = Path("/kaggle/working/diagnostics")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLOURS = {
    "bz":"#e63946","b_mag":"#457b9d","clock_angle":"#f4a261",
    "dbz_dt":"#2a9d8f","b_rotation":"#8338ec","bz_smoothed":"#ff006e",
    "bz_persistence":"#fb8500","b_elevation":"#06d6a0",
}
THRESH_LINES = {
    "bz":[("Bz=−10",-10,"--",0.5),("Bz=−20",-20,":",0.7)],
    "bz_smoothed":[("Bz_sm=−10",-10,"--",0.5)],
}

# ── Load data ─────────────────────────────────────────────────────────────────
logger.info("Loading MAG data...")
mag_raw = load_mag_directory(MAG_DIR)
mag_df  = resample_mag_to_1min(mag_raw).dropna(how="all")
feat_df = build_mag_features(mag_df)
feat_df = attach_labels(feat_df, isro_csv=None,
                        donki_start="2024-05-01", donki_end="2026-05-01",
                        use_known=True)
feat_df.dropna(subset=["bz","b_mag"], inplace=True)
X_raw, y = build_sequences(feat_df, seq_len=SEQUENCE_LENGTH, stride=STRIDE)

scaler  = joblib.load(SCALER_PATH)
n, t, f = X_raw.shape
X_sc    = scaler.transform(X_raw.reshape(-1,f)).reshape(n,t,f)

# ── Load model ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = PatchTransformer(input_dim=len(MAG_FEATURE_NAMES), seq_len=SEQUENCE_LENGTH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval().to(device)

# ── Inference on positives ────────────────────────────────────────────────────
pos_idx = np.where(y > 0)[0]
with torch.no_grad():
    probs = torch.sigmoid(
        model(torch.tensor(X_sc[pos_idx], dtype=torch.float32, device=device))
    ).cpu().numpy()

missed_local  = np.where(probs < THRESHOLD)[0]
missed_global = pos_idx[missed_local]
logger.info("Missed: %d / %d", len(missed_global), len(pos_idx))

# ── Classify ──────────────────────────────────────────────────────────────────
def classify(win):
    bz  = win[:, MAG_FEATURE_NAMES.index("bz")]
    mag = win[:, MAG_FEATURE_NAMES.index("b_mag")]
    rot = win[:, MAG_FEATURE_NAMES.index("b_rotation")]
    nan_frac  = np.isnan(win).mean()
    bz_min    = np.nanmin(bz)
    mag_max   = np.nanmax(mag)
    rot_total = np.nansum(np.abs(rot))
    reasons   = []
    if nan_frac > 0.15: reasons.append(f"DATA GAP ({nan_frac*100:.0f}% NaN)")
    if bz_min > -10:    reasons.append(f"WEAK BZ (min={bz_min:.1f} nT)")
    if mag_max < 15:    reasons.append(f"LOW |B| (max={mag_max:.1f} nT)")
    if rot_total < 5:   reasons.append(f"NO ROTATION (rot={rot_total:.2f})")
    if not reasons:     reasons.append("AMBIGUOUS — moderate CME near threshold")
    return dict(nan_frac=nan_frac, bz_min=bz_min, mag_max=mag_max,
                rot_total=rot_total,
                below_10=int((bz<-10).sum()), below_20=int((bz<-20).sum()),
                reasons=reasons)

# ── Plot ──────────────────────────────────────────────────────────────────────
def plot_window(win_raw, prob, diag, tag, out_path):
    t_hr = np.arange(win_raw.shape[0]) / 60.0
    fig  = plt.figure(figsize=(18,9))
    fig.patch.set_facecolor("#0f1117")
    fig.suptitle(f"{tag}  |  P={prob:.4f}  |  {' | '.join(diag['reasons'])}",
                 color="white", fontsize=10, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(2,4,figure=fig,hspace=0.45,wspace=0.35,
                           left=0.07,right=0.97,top=0.91,bottom=0.08)
    for i, fname in enumerate(MAG_FEATURE_NAMES):
        ax = fig.add_subplot(gs[i//4, i%4])
        ax.set_facecolor("#1a1d27")
        c  = FEATURE_COLOURS.get(fname,"#aaa")
        v  = win_raw[:, i]
        ax.plot(t_hr, v, color=c, lw=1.4)
        ax.fill_between(t_hr, v, alpha=0.15, color=c)
        for lbl,val,ls,a in THRESH_LINES.get(fname,[]):
            ax.axhline(val, color="white", ls=ls, lw=0.8, alpha=a)
        ax.axhline(0, color="#444", lw=0.6)
        ax.set_title(fname, color="white", fontsize=9, pad=3)
        ax.tick_params(colors="white", labelsize=7)
        for sp in ax.spines.values(): sp.set_edgecolor("#333")
        ax.set_xlabel("time (hr)", color="#aaa", fontsize=7)
        ax.text(0.98,0.97,f"min={np.nanmin(v):.2f}\nmax={np.nanmax(v):.2f}",
                transform=ax.transAxes,color="#ccc",fontsize=6.5,
                va="top",ha="right",
                bbox=dict(boxstyle="round,pad=0.2",fc="#222",ec="none",alpha=0.7))
    fig.text(0.005,0.50,
             f"Bz min:  {diag['bz_min']:.1f} nT\n"
             f"|B| max: {diag['mag_max']:.1f} nT\n"
             f"Bz<−10: {diag['below_10']} steps\n"
             f"Bz<−20: {diag['below_20']} steps\n"
             f"b_rot:   {diag['rot_total']:.2f}\n"
             f"NaN:     {diag['nan_frac']*100:.1f}%\n"
             f"P(CME):  {prob:.4f}",
             color="#ddd",fontsize=8,va="center",fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.4",fc="#1a1d27",ec="#444",alpha=0.9),
             transform=fig.transFigure)
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#0f1117")
    plt.close(fig)
    logger.info("Saved: %s", out_path)

# ── Run ───────────────────────────────────────────────────────────────────────
report = []
for rank, (li, gi) in enumerate(
    sorted(zip(missed_local, missed_global), key=lambda x: probs[x[0]])
):
    prob = float(probs[li])
    diag = classify(X_raw[gi])
    report.append(
        f"Missed #{rank+1} | idx={gi} | P={prob:.4f}\n"
        f"  Bz min={diag['bz_min']:.1f} nT | |B| max={diag['mag_max']:.1f} nT | "
        f"Bz<-10: {diag['below_10']} steps | rot={diag['rot_total']:.2f}\n"
        f"  Diagnosis: {' | '.join(diag['reasons'])}"
    )
    plot_window(X_raw[gi], prob, diag, f"MISSED CME #{rank+1}",
                OUT_DIR / f"missed_{rank+1}_idx{gi}.png")

# Hardest true positive
caught = np.where(probs >= THRESHOLD)[0]
if len(caught):
    hi   = caught[np.argmin(probs[caught])]
    prob = float(probs[hi])
    diag = classify(X_raw[pos_idx[hi]])
    report.append(
        f"\nHardest TP | idx={pos_idx[hi]} | P={prob:.4f}\n"
        f"  Bz min={diag['bz_min']:.1f} nT | |B| max={diag['mag_max']:.1f} nT\n"
        f"  Diagnosis: {' | '.join(diag['reasons'])}"
    )
    plot_window(X_raw[pos_idx[hi]], prob, diag, "HARDEST TRUE POSITIVE",
                OUT_DIR / f"hardest_tp_idx{pos_idx[hi]}.png")

print("\n" + "="*60)
print("MISSED WINDOW DIAGNOSTIC REPORT")
print("="*60)
for line in report: print(line)
print("="*60)
print(f"\nPlots saved to: {OUT_DIR}")

2026-05-15 08:12:08,654 INFO Loading MAG data...
2026-05-15 08:12:09,219 INFO Parsed: L2_AL1_MAG_20240509_V00.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 08:12:09,356 INFO Parsed: L2_AL1_MAG_20240510_V00.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 08:12:09,499 INFO Parsed: L2_AL1_MAG_20240511_V00.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 08:12:09,628 INFO Parsed: L2_AL1_MAG_20240512_V00.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 08:12:09,777 INFO Parsed: L2_AL1_MAG_20240513_V00.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 08:12:09,935 INFO Parsed: L2_AL1_MAG_20241214_V00.nc | 8640 rows | Bz [-8.0, 10.6] nT
2026-05-15 08:12:10,109 INFO Parsed: L2_AL1_MAG_20241215_V00.nc | 8640 rows | Bz [-7.7, 8.3] nT
2026-05-15 08:12:10,283 INFO Parsed: L2_AL1_MAG_20241216_V00.nc | 8640 rows | Bz [-6.7, 8.0] nT
2026-05-15 08:12:10,419 INFO Parsed: L2_AL1_MAG_20241217_V00.nc | 8640 rows | Bz [-21.0, 29.0] nT
2026-05-15 08:12:10,552 INFO Parsed: L2_AL1_MAG_20241218_V00.nc | 8640 rows | 

Feature matrix: (120269, 8) | columns: ['bz', 'b_mag', 'clock_angle', 'dbz_dt', 'b_rotation', 'bz_smoothed', 'bz_persistence', 'b_elevation']


2026-05-15 08:12:53,457 WARNING DONKI fetch failed: HTTPSConnectionPool(host='kauai.ccmc.gsfc.nasa.gov', port=443): Read timed out. (read timeout=30) -- using empty catalog
/kaggle/working/geomag/label_events.py:157: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_events = pd.concat(catalogs, ignore_index=True)
2026-05-15 08:12:53,467 INFO Labels: 5446 positive rows (4.53% of 120269 total)
2026-05-15 08:12:53,513 INFO Sequences: 7509 | shape: (7509, 128, 8) | CME rate: 4.85%
/kaggle/working/geomag/model_factory.py:341: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer,
2026-05-15 08:12:54,419 INFO Missed: 24 / 36


MISSED WINDOW DIAGNOSTIC REPORT
Missed #1 | idx=274 | P=0.1213
  Bz min=5.2 nT | |B| max=14.1 nT | Bz<-10: 0 steps | rot=25362.18
  Diagnosis: WEAK BZ (min=5.2 nT) | LOW |B| (max=14.1 nT)
Missed #2 | idx=4966 | P=0.1971
  Bz min=-5.7 nT | |B| max=24.1 nT | Bz<-10: 0 steps | rot=18059.88
  Diagnosis: WEAK BZ (min=-5.7 nT)
Missed #3 | idx=267 | P=0.6538
  Bz min=8.6 nT | |B| max=16.6 nT | Bz<-10: 0 steps | rot=31487.78
  Diagnosis: WEAK BZ (min=8.6 nT)
Missed #4 | idx=249 | P=0.6830
  Bz min=-3.5 nT | |B| max=17.8 nT | Bz<-10: 0 steps | rot=79215.53
  Diagnosis: WEAK BZ (min=-3.5 nT)
Missed #5 | idx=268 | P=0.6880
  Bz min=8.6 nT | |B| max=16.5 nT | Bz<-10: 0 steps | rot=31256.87
  Diagnosis: WEAK BZ (min=8.6 nT)
Missed #6 | idx=266 | P=0.6935
  Bz min=7.6 nT | |B| max=16.6 nT | Bz<-10: 0 steps | rot=30877.24
  Diagnosis: WEAK BZ (min=7.6 nT)
Missed #7 | idx=4994 | P=0.7268
  Bz min=-1.7 nT | |B| max=26.0 nT | Bz<-10: 0 steps | rot=9525.09
  Diagnosis: WEAK BZ (min=-1.7 nT)
Missed #8 | 

In [7]:
!python /kaggle/working/geomag/train.py \
    --mag_dir "/kaggle/input/datasets/aurachan/updated-windows-mag-data" \
    --epochs 100

2026-05-15 10:45:49,279 INFO Loading MAG data from: /kaggle/input/datasets/aurachan/updated-windows-mag-data
2026-05-15 10:45:50,261 INFO Parsed: L2_AL1_MAG_20240509_V00.nc | 8640 rows | Bz [-3.4, 7.1] nT
2026-05-15 10:45:50,382 INFO Parsed: L2_AL1_MAG_20240510_V00.nc | 8640 rows | Bz [-54.3, 68.8] nT
2026-05-15 10:45:50,506 INFO Parsed: L2_AL1_MAG_20240511_V00.nc | 8640 rows | Bz [-44.6, 34.6] nT
2026-05-15 10:45:50,613 INFO Parsed: L2_AL1_MAG_20240512_V00.nc | 8640 rows | Bz [-6.7, 14.9] nT
2026-05-15 10:45:50,723 INFO Parsed: L2_AL1_MAG_20240513_V00.nc | 8640 rows | Bz [-2.1, 15.1] nT
2026-05-15 10:45:50,836 INFO Parsed: L2_AL1_MAG_20241214_V00.nc | 8640 rows | Bz [-8.0, 10.6] nT
2026-05-15 10:45:50,946 INFO Parsed: L2_AL1_MAG_20241215_V00.nc | 8640 rows | Bz [-7.7, 8.3] nT
2026-05-15 10:45:51,057 INFO Parsed: L2_AL1_MAG_20241216_V00.nc | 8640 rows | Bz [-6.7, 8.0] nT
2026-05-15 10:45:51,178 INFO Parsed: L2_AL1_MAG_20241217_V00.nc | 8640 rows | Bz [-21.0, 29.0] nT
2026-05-15 10:45:5

In [18]:
print(f"Timestamp of Max Rotation: {max_idx}")

# If you want to see the raw magnetic values at that exact moment:
print("\n--- Raw MAG Values at Peak Rotation ---")
print(raw_df.loc[max_idx])

Timestamp of Max Rotation: 2024-05-10 16:44:50

--- Raw MAG Values at Peak Rotation ---
bx      -15.369565
by        4.769050
bz        0.928706
b_mag    16.119239
Name: 2024-05-10 16:44:50, dtype: float64
